# W4D5 — Fine-Tuning ResNet-18 — Guided

**Week 4 · Day 5 · CNNs & Model Fine-Tuning** · Lab

Yesterday's best number was 0.613, from a network trained from scratch on 316 images. Today a model
that has never seen your dataset — but has seen a million other photographs — passes that in its
**first epoch**, training 2,565 parameters instead of eleven million.

Four configurations on identical data, four curves on one figure:

- **A — from scratch.** `resnet18(weights=None)`, everything trainable. The honest control.
- **B — frozen backbone, new head.** 2,565 trainable parameters, and the number that makes the case.
- **C — unfreeze the last block, learning rate ÷ 10.** The standard recipe.
- **D — the mistake.** The same unfreezing at a learning rate chosen as if the weights were random.
  It is mandatory, it fails, and the failure is the most useful thing in the notebook.

Then the honest check: 25 images the model has never seen, **three of which are of a class nobody
trained on**. What a good service does with those is the last question of the week.

<div dir="rtl" align="right">

# الأسبوع ٤ · اليوم ٥ — الضبط الدقيق لـResNet-18

**الأسبوع الرابع · اليوم الخامس · الشبكات الالتفافية وضبط النماذج** · معمل

كان أفضل رقم بالأمس ٠٫٦١٣، من شبكة دُرِّبت من الصفر على ٣١٦ صورة. واليوم يتجاوزه في **حقبته الأولى**
نموذجٌ لم يرَ بياناتك قط، لكنه رأى مليون صورة أخرى، وهو يُدرّب ٢٬٥٦٥ معاملًا لا أحد عشر مليونًا.

أربعة إعدادات على البيانات نفسها، وأربعة منحنيات في شكل واحد:

- **أ — من الصفر.** `resnet18(weights=None)` وكل شيء قابل للتدريب. وهو الضابط الصادق.
- **ب — عمود فقري مجمَّد ورأس جديد.** ٢٬٥٦٥ معاملًا قابلًا للتدريب، وهو الرقم الذي يبني الحجّة.
- **ج — أطلِق الكتلة الأخيرة ومعدّل التعلّم ÷ ١٠.** الوصفة المعيارية.
- **د — الخطأ.** الإطلاق نفسه بمعدّل تعلّم اختير كأن الأوزان عشوائية. وهو إلزامي، ويفشل، وفشله أنفع ما
  في الدفتر.

ثم التحقّق الصادق: خمس وعشرون صورة لم يرها النموذج، **ثلاث منها من فئة لم يدرّبها أحد**. وما تفعله
الخدمة الجيّدة بها هو سؤال الأسبوع الأخير.

</div>

> **This is the guided version.** Most of the code is already here. Fill in the lines marked
> `# TODO`. If you want the full challenge, use the `_blank` version instead.

<div dir="rtl" align="right">

> **هذه النسخة الموجَّهة.** معظم الشيفرة موجودة، وعليك إكمال الأسطر المعلَّمة بـ `# TODO`.
> وإذا أردت التحدّي الكامل فاستخدم نسخة `_blank`.

</div>

## Learning objectives

By the end of this lab you can:

- Load a pretrained `torchvision` model, replace its head for your own classes, and say how many
  parameters that leaves trainable.
- Explain why the pretrained model's own normalisation statistics must be used, not your dataset's.
- Freeze and unfreeze parts of a network deliberately, and attach a learning-rate schedule.
- Say why fine-tuning uses a smaller learning rate than training from scratch, having watched a
  larger one destroy the thing you were standing on.
- Compare four runs on accuracy, trainable parameters and wall-clock time, and defend a choice.
- Report what a model should do with an input from a class it was never trained on.

<div dir="rtl" align="right">

## أهداف التعلّم

في نهاية هذا المعمل تستطيع:

- أن تُحمّل نموذجًا مُدرَّبًا مسبقًا من `torchvision`، وتستبدل رأسه بفئاتك، وتقول كم معاملًا يبقى قابلًا
  للتدريب.
- أن تشرح لماذا يجب استخدام إحصاءات التوحيد الخاصة بالنموذج المُدرَّب لا إحصاءات بياناتك.
- أن تُجمّد أجزاء من شبكة وتُطلقها عن قصد، وتُلحق بها جدولًا لمعدّل التعلّم.
- أن تقول لماذا يستخدم الضبط الدقيق معدّل تعلّم أصغر من التدريب من الصفر، بعد أن رأيت معدّلًا أكبر
  يُتلف ما كنت تقف عليه.
- أن تقارن أربع تشغيلات في الدقة وعدد المعاملات وزمن التدريب، وتدافع عن اختيار.
- أن تقول ما ينبغي أن يفعله النموذج بمُدخل من فئة لم يُدرَّب عليها قط.

</div>

## About the data

**Datasets:** `small_image_5class` — yesterday's 400 images, the same five classes and, crucially,
**the same split** — and `unseen_images`, 25 images held out from the same COCO pool.

The split is rebuilt here with the same duplicate-grouping and the same seed as W4D4, so today's
numbers sit directly next to yesterday's. If you change the seed, you lose the comparison that this
lab exists to make.

`unseen_images` ships a `labels.csv` with a column `in_or_out`. **Three of the 25 are elephants** —
a class that appears in no training set this week. They are there because a deployed classifier is
not asked polite questions: it is handed whatever the user uploads, and a five-class softmax will
always return five numbers that sum to one, however alien the input.

**First run downloads** the ImageNet weights for `resnet18` — 44.7 MB, cached afterwards. Total
training time for all four runs is about **2.7 minutes** on a laptop CPU (A 52 s, B 26 s, C 33 s,
D 51 s on the reference machine), and the stretch section adds about three minutes more.

<div dir="rtl" align="right">

## عن البيانات

**مجموعتا البيانات:** `small_image_5class` — صور الأمس الأربعمئة، الفئات الخمس نفسها، والأهمّ:
**التقسيم نفسه** — و`unseen_images`، خمس وعشرون صورة محجوزة من مجمّع COCO نفسه.

ويُعاد بناء التقسيم هنا بتجميع المكرّرات نفسه والبذرة نفسها كما في اليوم الرابع، فتقع أرقام اليوم
مباشرةً بجانب أرقام الأمس. وإن غيّرت البذرة خسرت المقارنة التي وُجد هذا المعمل لأجلها.

ويحمل `unseen_images` ملف `labels.csv` فيه عمود `in_or_out`. و**ثلاث من الخمس والعشرين أفيال** — من
فئة لا ترد في أي مجموعة تدريب هذا الأسبوع. وهي هناك لأن المصنّف المنشور لا تُطرح عليه أسئلة مهذّبة،
بل يُسلَّم ما يرفعه المستخدم، وستُرجع دالة softmax بخمس فئات خمسة أعداد مجموعها واحد مهما كان المُدخل
غريبًا.

**والتشغيل الأول يُنزّل** أوزان ImageNet لـ`resnet18`، أي ٤٤٫٧ ميغابايت، وتُخزَّن بعدها. وزمن التدريب
الكلّي للتشغيلات الأربع نحو **دقيقتين وأربعين ثانية** على معالج حاسوب محمول (أ ٥٢ ثانية، ب ٢٦، ج ٣٣،
د ٥١ على الجهاز المرجعي)، ويضيف القسم الإضافي نحو ثلاث دقائق.

</div>

## Setup

<div dir="rtl" align="right">

## الإعداد

</div>

In [ ]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, device, versions
from aiep.data import get_dataset_dir, load_artefact, describe_dataset
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, check_close, check_shape, report

ensure("torchinfo", "scikit-learn", "matplotlib")
seed_everything(42)

import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

IMAGE_ROOT = get_dataset_dir("small_image_5class") / "images"
UNSEEN_DIR = get_dataset_dir("unseen_images")
IMAGE_SIZE, EPOCHS, BATCH_SIZE, SEED = 160, 8, 32, 42

catalogue = datasets.ImageFolder(IMAGE_ROOT)
CLASSES = catalogue.classes
FILES = [path for path, _ in catalogue.samples]
LABELS = np.array([label for _, label in catalogue.samples])

print(describe_dataset("unseen_images"))
print(f"\n{len(FILES)} images | classes {CLASSES}")
print(versions(), "| device:", device())

In [ ]:
# Yesterday's split, rebuilt exactly: duplicates grouped, then the groups split, seed 42.
from sklearn.model_selection import train_test_split


def dhash(path, size=8):
    grey = Image.open(path).convert("L").resize((size + 1, size), Image.BICUBIC)
    pixels = np.asarray(grey, dtype=np.int16)
    return (pixels[:, 1:] > pixels[:, :-1]).flatten()


hashes = np.array([dhash(path) for path in FILES])
distance = (hashes[:, None, :] != hashes[None, :, :]).sum(-1)
np.fill_diagonal(distance, hashes.shape[1] + 1)
group = np.arange(len(FILES))
for left, right in [(int(i), int(j)) for i, j in zip(*np.where(distance <= 8)) if i < j]:
    group[group == group[right]] = group[left]

group_ids = np.unique(group)
group_labels = np.array([LABELS[group == g][0] for g in group_ids])
train_groups, val_groups = train_test_split(group_ids, test_size=0.2,
                                            stratify=group_labels, random_state=SEED)
TRAIN_INDEX = np.flatnonzero(np.isin(group, train_groups))
VAL_INDEX = np.flatnonzero(np.isin(group, val_groups))
print(f"train {len(TRAIN_INDEX)} | val {len(VAL_INDEX)} — the same split as W4D4")

## Section 1 — Warm-up: 11.2 million against 2,565  (≈25 min)

Everything here works. Load `resnet18` with its ImageNet weights, print the summary, then freeze the
backbone and replace the head — and print the summary again.

**11,689,512 parameters as it ships**, because its head is `Linear(512, 1000)` for ImageNet's
thousand categories. Replace that with `Linear(512, 5)` and the model holds **11,179,077**, of which
**2,565 are trainable**: 512 × 5 weights plus 5 biases. Everything else — every convolution that learned what an edge, a texture, a
wheel and an eye look like, from 1.2 million photographs — is frozen and is not going to change.

Then the normalisation. The pretrained weights expect inputs standardised with **ImageNet's** mean
and standard deviation, because that is the distribution every one of those 1.2 million images was
fed in as. Substitute your own dataset's statistics and every layer sees inputs shifted away from
what it was tuned for; the model still runs, and quietly does worse. `weights.transforms()` carries
the right numbers, so ask the weights rather than typing them.

<div dir="rtl" align="right">

## القسم الأول — الإحماء: ١١٫٢ مليونًا مقابل ٢٬٥٦٥ (نحو ٢٥ دقيقة)

كل ما هنا يعمل. حمّل `resnet18` بأوزان ImageNet واطبع الملخّص، ثم جمّد العمود الفقري واستبدل الرأس —
واطبع الملخّص ثانيةً.

**١١٬٦٨٩٬٥١٢ معاملًا كما يأتي**، لأن رأسه `Linear(512, 1000)` لفئات ImageNet الألف. واستبدله
بـ`Linear(512, 5)` فيصير النموذج **١١٬١٧٩٬٠٧٧**، منها **٢٬٥٦٥ قابلة للتدريب**: ٥١٢ × ٥ وزنًا مع
خمسة انحيازات. وكل ما عداه — كل التفاف تعلّم شكل الحافة والقوام والعجلة والعين من ١٫٢ مليون صورة — مجمَّد
ولن يتغيّر.

ثم التوحيد. فالأوزان المُدرَّبة تتوقّع مُدخلات موحَّدة بمتوسّط **ImageNet** وانحرافها المعياري، لأن ذلك
هو التوزيع الذي دخلت به كل صورة من تلك المليون والمئتي ألف. وإن وضعت إحصاءات بياناتك مكانها رأت كل
طبقة مُدخلات مزاحةً عمّا ضُبطت له؛ فيعمل النموذج ويسوء أداؤه بصمت. ويحمل `weights.transforms()`
الأرقام الصحيحة، فاسأل الأوزان بدل أن تكتبها.

</div>

In [ ]:
from torchinfo import summary

WEIGHTS = models.ResNet18_Weights.IMAGENET1K_V1
pretrained = models.resnet18(weights=WEIGHTS)
TOTAL_PARAMS = sum(p.numel() for p in pretrained.parameters())
print(f"resnet18, ImageNet weights: {TOTAL_PARAMS:,} parameters, "
      f"{sum(p.numel() for p in pretrained.parameters() if p.requires_grad):,} trainable")

for parameter in pretrained.parameters():
    parameter.requires_grad = False
torch.manual_seed(SEED)
pretrained.fc = nn.Linear(pretrained.fc.in_features, len(CLASSES))

HEAD_PARAMS = sum(p.numel() for p in pretrained.parameters() if p.requires_grad)
FIVE_CLASS_PARAMS = sum(p.numel() for p in pretrained.parameters())
print(f"backbone frozen, new head:  {sum(p.numel() for p in pretrained.parameters()):,} parameters, "
      f"{HEAD_PARAMS:,} trainable")
print(f"                            = {pretrained.fc.in_features} x {len(CLASSES)} weights + "
      f"{len(CLASSES)} biases")

preset = WEIGHTS.transforms()
print(f"\nthe weights' own normalisation: mean {preset.mean}, std {preset.std}")
print("those are ImageNet's statistics, not this dataset's — and they are the ones to use.")

In [ ]:
NORMALISE = transforms.Normalize(preset.mean, preset.std)

VAL_TRANSFORM = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    NORMALISE,
])

# The augmentation that won yesterday's A/B, unchanged.
TRAIN_TRANSFORM = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    NORMALISE,
])

train_loader = DataLoader(Subset(datasets.ImageFolder(IMAGE_ROOT, transform=TRAIN_TRANSFORM),
                                 TRAIN_INDEX), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(Subset(datasets.ImageFolder(IMAGE_ROOT, transform=VAL_TRANSFORM),
                               VAL_INDEX), batch_size=64)

# Yesterday's numbers, for the comparison in task 2.5.
aug_ab = pd.read_parquet(load_artefact("aug_ab.parquet"))
D4_BEST = float(aug_ab.val_accuracy.max())
print(aug_ab[["run", "val_accuracy"]].to_string(index=False))
print(f"\nyesterday's best from-scratch score: {D4_BEST:.3f} — that is the bar")

## Section 2 — Core: six tasks  (≈60 min)

1. Run A — from scratch, the control.
2. Run B — frozen backbone. Watch epoch 1.
3. Run C — unfreeze `layer4`, learning rate ÷ 10, with a schedule.
4. **Run D — the mistake.** Mandatory.
5. Four curves, a results table, and yesterday's number in it.
6. `unseen_images`, including the three the model cannot possibly know.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي: ست مهام (نحو ٦٠ دقيقة)

١. التشغيلة أ — من الصفر، وهي الضابط.
٢. التشغيلة ب — عمود فقري مجمَّد. راقب الحقبة الأولى.
٣. التشغيلة ج — أطلِق `layer4` ومعدّل التعلّم ÷ ١٠ مع جدول.
٤. **التشغيلة د — الخطأ.** إلزامية.
٥. أربعة منحنيات وجدول نتائج ورقم الأمس فيه.
٦. `unseen_images` ومنها الثلاث التي لا يمكن للنموذج أن يعرفها.

</div>

### Task 2.1 — the trainer, and run A from scratch

One function, four calls. It takes a model and a learning rate, trains for `EPOCHS`, and returns a
per-epoch DataFrame plus the wall-clock time — the time matters, because "as accurate and four times
faster" is an argument and "as accurate" alone is not.

Run A is `resnet18(weights=None)`: the identical architecture with random weights, everything
trainable, at 1e-3. This is the control that makes the rest of the notebook mean something. It gets
the same augmentation, the same split, the same epochs as every other run.

On the reference machine it ends at 0.425 with a best of **0.675** across the eight epochs — eleven
million parameters, 316 images, and a curve that jumps around because there is nothing holding it
steady.

<div dir="rtl" align="right">

### المهمة ٢٫١ — المُدرِّب، والتشغيلة أ من الصفر

دالة واحدة وأربعة نداءات. تأخذ نموذجًا ومعدّل تعلّم، وتُدرّب `EPOCHS` حقبة، وتُرجع `DataFrame` لكل
حقبة مع زمن الساعة — والزمن مهمّ، لأن «بالدقة نفسها وأسرع أربع مرّات» حجّة، و«بالدقة نفسها» وحدها ليست
حجّة.

والتشغيلة أ هي `resnet18(weights=None)`: البنية المطابقة بأوزان عشوائية، وكل شيء قابل للتدريب، عند
١e−٣. وهي الضابط الذي يجعل لبقيّة الدفتر معنى. وتأخذ الزيادة نفسها والتقسيم نفسه والحقب نفسها كأي
تشغيلة أخرى.

وعلى الجهاز المرجعي تنتهي عند ٠٫٤٢٥ وأفضلها **٠٫٦٧٥** في الحقب الثماني — أحد عشر مليون معامل، وثلاثمئة
وستّ عشرة صورة، ومنحنى يقفز لأن لا شيء يُثبّته.

</div>

In [ ]:


def build(pretrained_weights=True, freeze_backbone=True):
    """A resnet18 with a fresh five-class head, seeded so every run starts identically."""
    # TODO: then replace fc with a fresh Linear for len(CLASSES) classes.
    # مهمة: استبدل `fc` بطبقة خطّية جديدة بعدد `len(CLASSES)` فئات.


def train(model, learning_rate, name, scheduler=None, epochs=EPOCHS):
    """Train one configuration and return its per-epoch history."""
    # TODO: train accuracy, validation accuracy and elapsed seconds.
    # مهمة: التدريب ودقة التحقّق والثواني المنقضية.


# TODO: Run A: resnet18 from scratch, nothing frozen, learning rate 1e-3.
# مهمة: التشغيلة أ: `resnet18` من الصفر، ولا شيء مجمَّد، ومعدّل التعلّم ١e−٣.

### Task 2.2 — run B, the frozen backbone

The same architecture, this time with the ImageNet weights, everything frozen except the new head.
2,565 trainable parameters, same learning rate, same everything else.

**Look at epoch 1 before you look at anything else.** On the reference run it scores 0.738 — higher
than run A reaches in eight epochs, and higher than yesterday's best from-scratch number. One epoch,
2,565 parameters, on a laptop CPU.

The reason is worth saying out loud: `resnet18`'s convolutions already extract features that
separate buses from cats, because separating a thousand ImageNet categories required exactly that.
Your five classes are a linear problem *in that feature space*, and the only thing left to learn is
which direction each class points.

Note the time too. Run B is faster than run A despite the same forward pass, because there are no
gradients to compute or store for the frozen 11.18 million.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — التشغيلة ب، العمود الفقري المجمَّد

البنية نفسها، لكن بأوزان ImageNet هذه المرّة، وكل شيء مجمَّد إلا الرأس الجديد. ٢٬٥٦٥ معاملًا قابلًا
للتدريب، ومعدّل التعلّم نفسه، وكل ما عداه نفسه.

**وانظر إلى الحقبة الأولى قبل أي شيء آخر.** ففي التشغيل المرجعي تبلغ ٠٫٧٣٨ — أعلى مما تبلغه أ في ثماني
حقب، وأعلى من أفضل رقم من الصفر بالأمس. حقبة واحدة، و٢٬٥٦٥ معاملًا، على معالج حاسوب محمول.

والسبب يستحقّ أن يُقال بصوت مسموع: التفافات `resnet18` تستخرج أصلًا سماتٍ تفصل الحافلات عن القطط، لأن
فصل ألف فئة في ImageNet تطلّب ذلك بالضبط. وفئاتك الخمس مسألة خطّية **في فضاء تلك السمات**، وكل ما بقي
تعلّمه أي اتّجاه تشير إليه كل فئة.

ولاحظ الزمن أيضًا. فالتشغيلة ب أسرع من أ رغم تطابق التمرير الأمامي، لأنه لا تدرّجات تُحسب أو تُخزَّن
للأحد عشر مليونًا ومئة وثمانين ألفًا المجمَّدة.

</div>

In [ ]:

# TODO: Run B: pretrained weights, frozen backbone, new head, same learning rate.
# مهمة: التشغيلة ب: أوزان مُدرَّبة مسبقًا، وعمود فقري مجمَّد، ورأس جديد، ومعدّل التعلّم نفسه.

print(f"\nrun B, epoch 1:  {run_b.val_accuracy.iloc[0]:.3f}")
print(f"run A, epoch {EPOCHS}:  {run_a.val_accuracy.iloc[-1]:.3f}   "
      f"(best it managed: {run_a.val_accuracy.max():.3f})")
print(f"trainable parameters: {run_b.trainable.iloc[0]:,} against {run_a.trainable.iloc[0]:,}")
print(f"wall clock: {run_b.seconds.iloc[-1]:.0f}s against {run_a.seconds.iloc[-1]:.0f}s")

### Task 2.3 — run C, unfreeze the last block at a tenth of the learning rate

Run B only ever adjusted a linear map on top of frozen features. Run C lets the **last** residual
block move too — `layer4`, the most abstract features, the ones most specific to ImageNet's
categories and therefore the ones most worth adapting to yours.

Two rules, and they are the whole recipe:

- **Unfreeze from the top.** Early layers hold edges and textures, which are the same for every
  photograph ever taken. Late layers hold "this looks like a kind of dog", which is where your
  dataset differs.
- **Divide the learning rate.** 1e-4, not 1e-3. These weights are already good; you are nudging
  them, not searching for them. Add a `StepLR` that divides by ten again every three epochs, so the
  nudges get smaller as the model settles.

It reaches 0.925 on the reference run — and so does B. Their **best** epochs are identical; C is
ahead only on the final epoch, 0.925 against 0.912, which is one validation image out of 80. Eight
and a half million trainable parameters bought one image, and the honest report says that rather
than claiming a win.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — التشغيلة ج، أطلِق الكتلة الأخيرة عند عُشر معدّل التعلّم

لم تكن التشغيلة ب تُعدّل إلا خريطة خطّية فوق سمات مجمَّدة. أما ج فتدع الكتلة المتبقّية **الأخيرة**
تتحرّك أيضًا — `layer4`، أي أكثر السمات تجريدًا، وأشدّها اختصاصًا بفئات ImageNet، فأجدرها بالتكيّف
مع فئاتك.

وقاعدتان هما الوصفة كلها:

- **أطلِق من الأعلى.** فالطبقات المبكّرة تحمل الحواف والقوام، وهي نفسها في كل صورة التُقطت يومًا.
  والطبقات المتأخّرة تحمل «هذا يشبه نوعًا من الكلاب»، وهنا تختلف بياناتك.
- **اقسم معدّل التعلّم.** ١e−٤ لا ١e−٣. فهذه الأوزان جيّدة أصلًا، وأنت تدفعها دفعًا خفيفًا لا تبحث
  عنها. وأضِف `StepLR` يقسم على عشرة كل ثلاث حقب، فتصغر الدفعات كلما استقرّ النموذج.

وتبلغ ٠٫٩٢٥ في التشغيل المرجعي، وتبلغها ب كذلك. فأفضل حقبةٍ فيهما متطابقة، ولا تتقدّم ج إلا في الحقبة
الأخيرة: ٠٫٩٢٥ مقابل ٠٫٩١٢، أي صورة واحدة من ثمانين. ثمانية ملايين ونصف معامل قابل للتدريب اشترت صورة
واحدة، والتقرير الصادق يقول ذلك بدل أن يدّعي فوزًا.

</div>

In [ ]:
from torch.optim.lr_scheduler import StepLR


def unfreeze_last_block(model):
    """Let layer4 train; everything before it stays frozen."""
    # TODO: Set requires_grad = True on layer4's parameters and return the model.
    # مهمة: اجعل `requires_grad = True` لمعاملات `layer4` وأرجِع النموذج.


# TODO: Run C: pretrained, layer4 unfrozen, lr 1e-4, StepLR every 3 epochs.
# مهمة: التشغيلة ج: مُدرَّب مسبقًا، و`layer4` مُطلَقة، ومعدّل ١e−٤، و`StepLR` كل ثلاث حقب.

gap_in_images = (run_c.val_accuracy.max() - run_b.val_accuracy.max()) * len(VAL_INDEX)
print(f"\nC best {run_c.val_accuracy.max():.3f} vs B best {run_b.val_accuracy.max():.3f} "
      f"— a difference of {gap_in_images:.0f} validation image(s)")

### Task 2.4 — run D, the mistake

Same as run C — pretrained weights, unfrozen — but at a learning rate chosen the way you would
choose one for random weights. This run is **mandatory**, and it is meant to fail.

The reference numbers, so you know what you are reproducing:

| what was unfrozen | learning rate | best validation |
|---|---|---|
| `layer4` | 1e-4 (run C) | 0.925 |
| `layer4` | 1e-3 (run A's) | 0.963 |
| everything | 1e-3 | 0.900 |
| **everything** | **1e-2** | **0.500** |

Read that table before you run anything, because it says something more useful than "use a small
learning rate". Unfreezing `layer4` at ten times the recommended rate did **not** break anything —
it was fine, even good. Unfreezing everything at ten times *that* destroyed the model: 0.500, worse
than run A managed from random weights, because the first gradient steps overwrote the ImageNet
features the whole method depends on before the head had learned anything to protect them.

**Run D is the last row.** It is what you do by accident when you copy a training script that was
written for from-scratch training and point it at a pretrained model: nothing frozen, an aggressive
learning rate, and a result that looks like the model simply is not very good.

The assert on this run passes when D scores **below** run A. Failing it means you did not manage to
break the model, which is a nice problem to have and still means the run is not showing the lesson.

<div dir="rtl" align="right">

### المهمة ٢٫٤ — التشغيلة د، الخطأ

كالتشغيلة ج — أوزان مُدرَّبة مسبقًا ومُطلَقة — لكن بمعدّل تعلّم اختير كما تختاره لأوزان عشوائية. وهذه
التشغيلة **إلزامية**، والمقصود منها أن تفشل.

والأرقام المرجعية لتعرف ما الذي تُعيد إنتاجه:

| ما أُطلق | معدّل التعلّم | أفضل تحقّق |
|---|---|---|
| `layer4` | ١e−٤ (التشغيلة ج) | ٠٫٩٢٥ |
| `layer4` | ١e−٣ (معدّل أ) | ٠٫٩٦٣ |
| كل شيء | ١e−٣ | ٠٫٩٠٠ |
| **كل شيء** | **١e−٢** | **٠٫٥٠٠** |

اقرأ الجدول قبل أن تشغّل شيئًا، فهو يقول ما هو أنفع من «استخدم معدّل تعلّم صغير». فإطلاق `layer4`
بعشرة أضعاف المعدّل الموصى به لم يكسر شيئًا — بل كان جيّدًا. أما إطلاق كل شيء بعشرة أضعاف **ذلك** فقد
أتلف النموذج: ٠٫٥٠٠، أسوأ مما بلغته أ من أوزان عشوائية، لأن خطوات التدرّج الأولى طمست سمات ImageNet
التي تقوم عليها الطريقة كلها قبل أن يتعلّم الرأس ما يحميها.

**والتشغيلة د هي الصف الأخير.** وهي ما تفعله بالخطأ حين تنسخ سكربت تدريب كُتب للتدريب من الصفر
وتوجّهه إلى نموذج مُدرَّب مسبقًا: لا شيء مجمَّد، ومعدّل تعلّم عدواني، ونتيجةٌ تبدو وكأن النموذج ليس
جيّدًا وحسب.

ويمرّ الفحص على هذه التشغيلة حين تسجّل **دون** التشغيلة أ. وسقوطه يعني أنك لم تفلح في كسر النموذج،
وهي مشكلة سارّة، لكنها تعني أيضًا أن التشغيلة لا تُظهر الدرس.

</div>

In [ ]:

# TODO: Run D: pretrained, everything unfrozen, learning rate 1e-2, no schedule.
# مهمة: التشغيلة د: مُدرَّب مسبقًا، وكل شيء مُطلَق، ومعدّل التعلّم ١e−٢، وبلا جدول.

print(f"\nD best {run_d.val_accuracy.max():.3f} | A best {run_a.val_accuracy.max():.3f} | "
      f"C best {run_c.val_accuracy.max():.3f}")

WHAT_BROKE = (
    # TODO: One sentence: what did the learning rate do to the pretrained weights?
    # مهمة: جملة واحدة: ما الذي فعله معدّل التعلّم بالأوزان المُدرَّبة مسبقًا؟
)
print("\n" + WHAT_BROKE)

### Task 2.5 — four curves, one table, and yesterday's number

All four validation curves on one figure, then a results table with the three columns that let
somebody else make the decision: final accuracy, trainable parameters, and seconds.

Add yesterday's best from `aug_ab.parquet` as a horizontal line. That number — a small CNN trained
from scratch with the winning augmentation — is the honest alternative to all of this, and run B
passes it in one epoch with 2,565 trainable parameters.

The row worth arguing about is B against C. C is not obviously better on 80 validation images, and
it trains 3,000× more parameters to get there. On this dataset, at this size, **B is the answer**,
and being able to say that — rather than reaching for the most elaborate option — is the skill.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — أربعة منحنيات وجدول واحد ورقم الأمس

منحنيات التحقّق الأربعة في شكل واحد، ثم جدول نتائج بالأعمدة الثلاثة التي تتيح لغيرك أن يتّخذ القرار:
الدقة النهائية، والمعاملات القابلة للتدريب، والثواني.

وأضِف أفضل رقم بالأمس من `aug_ab.parquet` خطًّا أفقيًا. فذلك الرقم — شبكة التفافية صغيرة دُرِّبت من
الصفر بالزيادة الفائزة — هو البديل الصادق لهذا كله، وتتجاوزه التشغيلة ب في حقبة واحدة بـ٢٬٥٦٥ معاملًا.

والصفّ الجدير بالجدل هو ب مقابل ج. فليست ج أفضل بوضوح على ثمانين صورة تحقّق، وهي تُدرّب ثلاثة آلاف ضعف
من المعاملات لتبلغ ذلك. وعلى هذه البيانات وبهذا الحجم **الجواب هو ب**، والقدرة على قول ذلك — بدل
التعلّق بأكثر الخيارات تعقيدًا — هي المهارة.

</div>

In [ ]:

# TODO: build the results table.
# مهمة: ارسم منحنيات التحقّق الأربعة مع أفضل رقم بالأمس خطًّا أفقيًا، ثم ابنِ جدول النتائج.

print(results.round(3).to_string(index=False))
print(f"\nyesterday's best from-scratch score: {D4_BEST:.3f}")
print(f"run B passes it at epoch "
      f"{int(run_b[run_b.val_accuracy > D4_BEST].epoch.min())}, training "
      f"{run_b.trainable.iloc[0]:,} parameters")

### Task 2.6 — the 25 images, and the three it cannot know

Take the winning model and predict on `unseen_images`. For each image record the predicted class and
the **maximum softmax probability** — the model's confidence in its own answer.

Then split those 25 by `labels.csv`'s `in_or_out` column and compare the two confidence
distributions. The 22 in-class images should mostly be confident and mostly right. The three
elephants cannot be right: there is no elephant output. The question is whether the model *hedges* —
returns a low maximum probability, which a service can act on — or whether it insists.

Plot both distributions and write one sentence in `DEPLOYMENT_NOTE`: what should a deployed service
do with an input it scores like that? "Reject below a threshold" is the beginning of an answer; say
what threshold, chosen how, and what happens to the in-class images you would reject along with it.

<div dir="rtl" align="right">

### المهمة ٢٫٦ — الصور الخمس والعشرون، والثلاث التي لا يمكن أن يعرفها

خُذ النموذج الفائز وتنبّأ على `unseen_images`. وسجّل لكل صورة الفئة المتوقَّعة و**أكبر احتمال softmax**،
أي ثقة النموذج في جوابه.

ثم افصل الخمس والعشرين بعمود `in_or_out` من `labels.csv` وقارن توزيعَي الثقة. فالصور الاثنتان
والعشرون داخل الفئات ينبغي أن تكون واثقة وصحيحة غالبًا. أما الأفيال الثلاثة فلا يمكن أن تكون صحيحة:
إذ لا مخرج للفيل. والسؤال هل **يتحفّظ** النموذج فيُرجع احتمالًا أقصى منخفضًا تستطيع الخدمة التصرّف
بناءً عليه، أم يُصرّ.

ارسم التوزيعين واكتب جملة في `DEPLOYMENT_NOTE`: ماذا ينبغي أن تفعل خدمة منشورة بمُدخل تُعطيه مثل هذه
الدرجة؟ و«ارفض ما دون عتبة» بداية جواب؛ فقل أي عتبة، واختيرت كيف، وماذا يحدث لصور الفئات التي سترفضها
معها.

</div>

In [ ]:
WINNER = run_b if run_b.val_accuracy.max() >= run_c.val_accuracy.max() else run_c
WINNING_NAME = WINNER["run"].iloc[0]
print(f"the winning configuration is {WINNING_NAME}")

WINNING_MODEL = model_b if WINNING_NAME.startswith("B") else model_c

# TODO: Predict on the 25 unseen images and record class and max softmax confidence.
# مهمة: تنبّأ على الصور الخمس والعشرين وسجّل الفئة وأكبر احتمال softmax.

in_class = unseen_labels[unseen_labels.in_or_out == "in"]
out_of_class = unseen_labels[unseen_labels.in_or_out == "out"]
OUT_MAX_CONFIDENCE = float(out_of_class.confidence.max())

print(f"\nin-class ({len(in_class)}): accuracy {in_class.correct.mean():.3f}, "
      f"mean confidence {in_class.confidence.mean():.3f}")
print(f"out-of-class ({len(out_of_class)}): mean confidence "
      f"{out_of_class.confidence.mean():.3f}, highest {OUT_MAX_CONFIDENCE:.3f}")
print(out_of_class[["file", "predicted", "confidence"]].round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist([in_class.confidence, out_of_class.confidence], bins=10, range=(0.2, 1.0),
        label=[f"in-class ({len(in_class)})", f"elephants ({len(out_of_class)})"],
        color=["#37a", "#c44"])
ax.set_xlabel("max softmax probability"); ax.set_ylabel("images")
ax.legend(); ax.set_title("confidence on classes it knows, and on one it does not")
plt.tight_layout(); plt.show()

DEPLOYMENT_NOTE = (
    # TODO: What should a deployed service do with these three, and at what threshold?
    # مهمة: ماذا ينبغي أن تفعل خدمة منشورة بهذه الثلاث، وعند أي عتبة؟
)
print("\n" + DEPLOYMENT_NOTE)

## Section 3 — Stretch: which backbone would you deploy?  (≈30 min)

Swap `resnet18` for `mobilenet_v3_small` and for `efficientnet_b0`, frozen-backbone configuration
each time — run B's recipe, which is the one that won. For each: validation accuracy, total
parameters, and inference time on the validation set.

Then answer the question a client actually asks. Which one goes on a phone, which one on a server,
and what did each choice cost? `mobilenet_v3_small` has about a fifth of `resnet18`'s parameters;
`efficientnet_b0` has about the same count but a much deeper graph and a slower forward pass on
CPU. There is no winner in the abstract — there is a winner given a latency budget, a memory budget
and an accuracy floor, and naming those three is the answer.

Only the head changes shape between the three: `mobilenet_v3_small`'s classifier is a `Sequential`
whose last layer is the `Linear`, and `efficientnet_b0`'s is the same idea. Look at the model before
you assume `.fc`.

<div dir="rtl" align="right">

## القسم الثالث — الإضافي: أي عمود فقري تنشر؟ (نحو ٣٠ دقيقة)

استبدل `resnet18` بـ`mobilenet_v3_small` ثم بـ`efficientnet_b0`، بإعداد العمود الفقري المجمَّد في كل
مرّة — أي وصفة التشغيلة ب التي فازت. ولكلٍّ منها: دقة التحقّق، وإجمالي المعاملات، وزمن الاستدلال على
مجموعة التحقّق.

ثم أجب عن السؤال الذي يطرحه العميل فعلًا. أيّها على الهاتف وأيّها على الخادم، وكم كلّف كل اختيار؟
فـ`mobilenet_v3_small` نحو خُمس معاملات `resnet18`، و`efficientnet_b0` قريب منه في العدد لكن رسمه
البياني أعمق وتمريره الأمامي أبطأ على المعالج. ولا فائز في المطلق — بل فائز بمعلومية ميزانية زمن
الاستجابة وميزانية الذاكرة وحدّ الدقة الأدنى، وتسمية هذه الثلاثة هي الجواب.

ولا يتغيّر بين الثلاثة إلا شكل الرأس: فمصنّف `mobilenet_v3_small` هو `Sequential` آخر طبقاته
`Linear`، ومثله `efficientnet_b0`. فانظر إلى النموذج قبل أن تفترض `.fc`.

</div>

In [ ]:

# TODO: accuracy, parameter count and inference time per image.
# مهمة: الاستدلال لكل صورة.

print(backbones.round(3).to_string(index=False))

DEPLOYMENT_CHOICE = (
    # TODO: Phone, server, and what each choice cost. Name the budgets, not just a winner.
    # مهمة: الهاتف والخادم وكم كلّف كل اختيار. وسمِّ الميزانيات لا الفائز وحده.
)
print("\n" + DEPLOYMENT_CHOICE)

## Save your artefacts

`finetuned.pt` — the winning state dict plus the configuration needed to rebuild it: architecture,
class names, image size, normalisation statistics, and the accuracy it earned. A state dict without
that configuration is a file nobody can load correctly six weeks from now, including you.

`transfer_curves.parquet` — every run, every epoch, with trainable parameters and seconds.

**This is the capstone template.** Any capstone with an image component starts here: run B's recipe,
this artefact shape, and the honest check on held-out images including ones from outside the label
set. Your proposal was due yesterday; this notebook is the answer to "how would I actually build
it".

<div dir="rtl" align="right">

## احفظ آثارك

`finetuned.pt` — قاموس حالة النموذج الفائز مع الإعدادات اللازمة لإعادة بنائه: البنية، وأسماء الفئات،
وحجم الصورة، وإحصاءات التوحيد، والدقة التي بلغها. فقاموس الحالة بلا هذه الإعدادات ملفٌّ لا يستطيع أحد
تحميله تحميلًا صحيحًا بعد ستة أسابيع، وأنت منهم.

`transfer_curves.parquet` — كل تشغيلة وكل حقبة، مع المعاملات القابلة للتدريب والثواني.

**وهذا قالب مشروع التخرّج.** فأي مشروع فيه مكوّن صوري يبدأ من هنا: وصفة التشغيلة ب، وشكل هذا الأثر،
والتحقّق الصادق على صور محجوزة فيها ما هو خارج مجموعة التسميات. وقد كان مقترحك مستحقًّا بالأمس، وهذا
الدفتر جواب سؤال «كيف أبنيه فعلًا».

</div>

In [ ]:
MODEL_PATH = ARTEFACT_DIR / "finetuned.pt"
CURVES_PATH = ARTEFACT_DIR / "transfer_curves.parquet"

torch.save({
    "state_dict": WINNING_MODEL.state_dict(),
    "architecture": "resnet18",
    "configuration": WINNING_NAME,
    "classes": CLASSES,
    "image_size": IMAGE_SIZE,
    "normalise_mean": list(preset.mean),
    "normalise_std": list(preset.std),
    "val_accuracy": float(WINNER.val_accuracy.max()),
    "trainable_params": int(WINNER.trainable.iloc[0]),
    "epochs": EPOCHS,
    "seed": SEED,
}, MODEL_PATH)

curves.to_parquet(CURVES_PATH, index=False)
unseen_labels.to_parquet(ARTEFACT_DIR / "unseen_predictions.parquet", index=False)

print(f"wrote {MODEL_PATH.name} ({MODEL_PATH.stat().st_size / 1e6:.1f} MB) and "
      f"{CURVES_PATH.name} ({len(curves)} rows)")
print(results.round(3).to_string(index=False))

## Sanity check

<div dir="rtl" align="right">

## فحص سلامة

</div>

In [ ]:
check(HEAD_PARAMS == 2565 and abs(FIVE_CLASS_PARAMS - 11_179_077) < 1000,
      f"the frozen model must have exactly 2,565 trainable parameters (512 x 5 + 5) out of the "
      f"11.2M it holds with a five-class head — got {HEAD_PARAMS:,} of {FIVE_CLASS_PARAMS:,}",
      f"يجب أن يملك النموذج المجمَّد ٢٬٥٦٥ معاملًا قابلًا للتدريب بالضبط (٥١٢ × ٥ + ٥) من ١١٫٢ "
      f"مليون بالرأس الخماسي — والناتج {HEAD_PARAMS:,} من {FIVE_CLASS_PARAMS:,}")

check(run_b.val_accuracy.iloc[0] > run_a.val_accuracy.iloc[-1],
      f"run B's FIRST epoch must beat run A's LAST — got {run_b.val_accuracy.iloc[0]:.3f} "
      f"against {run_a.val_accuracy.iloc[-1]:.3f}",
      f"يجب أن تتجاوز الحقبة **الأولى** في ب الحقبة **الأخيرة** في أ — والناتج "
      f"{run_b.val_accuracy.iloc[0]:.3f} مقابل {run_a.val_accuracy.iloc[-1]:.3f}")

check(run_c.val_accuracy.max() >= run_b.val_accuracy.max(),
      f"run C should not score below run B — got {run_c.val_accuracy.max():.3f} against "
      f"{run_b.val_accuracy.max():.3f}",
      f"لا ينبغي أن تسجّل ج دون ب — والناتج {run_c.val_accuracy.max():.3f} مقابل "
      f"{run_b.val_accuracy.max():.3f}")

check(run_d.val_accuracy.max() < run_a.val_accuracy.max(),
      f"run D must score BELOW run A — passing this check means you reproduced the mistake "
      f"correctly: a pretrained model destroyed by too large a learning rate does worse than "
      f"random weights. Got D {run_d.val_accuracy.max():.3f}, A {run_a.val_accuracy.max():.3f}",
      f"يجب أن تسجّل د **دون** أ — واجتياز هذا الفحص يعني أنك أعدت إنتاج الخطأ إعادةً صحيحة: "
      f"فالنموذج المُدرَّب مسبقًا الذي أتلفه معدّل تعلّم كبير يكون أسوأ من الأوزان العشوائية. "
      f"والناتج د {run_d.val_accuracy.max():.3f} وأ {run_a.val_accuracy.max():.3f}")

check(WINNER.val_accuracy.max() > D4_BEST,
      f"the winning transfer run must beat yesterday's best from-scratch score — got "
      f"{WINNER.val_accuracy.max():.3f} against W4D4's {D4_BEST:.3f}",
      f"يجب أن تتجاوز تشغيلة النقل الفائزة أفضل نتيجة من الصفر بالأمس — والناتج "
      f"{WINNER.val_accuracy.max():.3f} مقابل {D4_BEST:.3f}")

check(len(unseen_labels) == 25 and unseen_labels.confidence.notna().all(),
      f"all 25 unseen images need a prediction and a confidence — got "
      f"{len(unseen_labels)} rows, {int(unseen_labels.confidence.isna().sum())} missing",
      f"تحتاج الصور الخمس والعشرون كلها تنبّؤًا وثقة — والموجود {len(unseen_labels)} صفًّا و"
      f"{int(unseen_labels.confidence.isna().sum())} ناقصًا")

check(OUT_MAX_CONFIDENCE < in_class.confidence.mean(),
      f"the three out-of-class images must be less confident than the in-class average — their "
      f"highest is {OUT_MAX_CONFIDENCE:.3f} against an in-class mean of "
      f"{in_class.confidence.mean():.3f}. If they are not, the model is confidently wrong about "
      f"an elephant and no threshold will save the service",
      f"يجب أن تكون الصور الثلاث خارج الفئات أقلّ ثقةً من متوسّط الفئات — فأعلاها "
      f"{OUT_MAX_CONFIDENCE:.3f} مقابل متوسّط {in_class.confidence.mean():.3f}. وإن لم تكن كذلك "
      f"فالنموذج واثق وهو مخطئ في فيل، ولن تنقذ الخدمةَ أي عتبة")

check(len(WHAT_BROKE.split()) >= 15 and len(DEPLOYMENT_NOTE.split()) >= 15,
      f"tasks 2.4 and 2.6 want written findings — got {len(WHAT_BROKE.split())} and "
      f"{len(DEPLOYMENT_NOTE.split())} words",
      f"تريد المهمّتان ٢٫٤ و٢٫٦ نتائج مكتوبة — والموجود {len(WHAT_BROKE.split())} و"
      f"{len(DEPLOYMENT_NOTE.split())} كلمة")

report()

## What's next

The week is done. You can compute a convolution by hand, build and train a CNN, evaluate a detector
properly, measure what augmentation bought you, and stand on a pretrained model instead of starting
from zero.

**Week 5** leaves images for text, and the shape of the argument repeats: a pretrained model that
has read a great deal, a small head for your task, and a small learning rate. What changes is the
data; run B's recipe does not.

**For the capstone**, this notebook is the template for any image component. `finetuned.pt` has the
shape week 8 loads for deployment, and `unseen_predictions.parquet` is the honest check that the
proposal's requirement 4 asks for, in miniature.

<div dir="rtl" align="right">

## ما التالي

انتهى الأسبوع. تستطيع الآن أن تحسب التفافًا بيدك، وأن تبني شبكة التفافية وتُدرّبها، وأن تُقيّم كاشفًا
تقييمًا صحيحًا، وأن تقيس ما أضافته زيادة البيانات، وأن تقف على نموذج مُدرَّب مسبقًا بدل أن تبدأ من الصفر.

و**الأسبوع الخامس** يترك الصور إلى النصوص، ويتكرّر شكل الحجّة نفسه: نموذج مُدرَّب مسبقًا قرأ كثيرًا،
ورأس صغير لمهمّتك، ومعدّل تعلّم صغير. والذي يتغيّر هو البيانات، أما وصفة التشغيلة ب فلا.

و**لمشروع التخرّج**، هذا الدفتر قالب أي مكوّن صوري. فـ`finetuned.pt` بالشكل الذي يُحمّله الأسبوع
الثامن للنشر، و`unseen_predictions.parquet` هو التحقّق الصادق الذي يطلبه المتطلّب الرابع في المقترح،
مصغَّرًا.

</div>